In [1]:
import torch
import torch.nn as nn
from config.schema import ArchitectureConfig
from utils.modules import get_module

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, config: ArchitectureConfig) -> None:
        super().__init__()
        self.layers: nn.ModuleList = nn.ModuleList()
        
        current_size: int = config.in_size
        for hidden_size in config.hidden_layers:
            self.layers.append(nn.Linear(current_size, hidden_size))
            self.layers.append(get_module(config.activation)())
            if config.use_dropout:
                self.layers.append(nn.Dropout(config.dropout, inplace=config.dropout_inplace))
            current_size = hidden_size
        
        self.layers.append(nn.Linear(current_size, config.out_size))

        if final_activation := config.final_activation:
            self.layers.append(get_module(final_activation)())
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

    def check_dying_relus(self, data: torch.Tensor, threshold: float = 0.0) -> dict[str, dict]:
        """
        Run a forward pass and report dying ReLU neurons per activation layer.

        Args:
            data: Input tensor (batch of samples, the larger the more reliable).
            threshold: Fraction of samples a neuron must be active for to be "alive".
            0.0 means a neuron is dead only if it fires for zero samples.

        Returns:
            Dict mapping layer name -> {total, dead, dead_fraction, dead_indices}
        """
        activations: dict[str, torch.Tensor] = {}
        hooks = []

        # Register hooks on ReLU (or LeakyReLU, etc.) layers
        for name, layer in self.layers.named_modules():
            if isinstance(layer, (nn.ReLU, nn.LeakyReLU, nn.PReLU, nn.ELU, nn.GELU)):
                def _hook(module, input, output, name=name):
                    activations[name] = output.detach()
                hooks.append(layer.register_forward_hook(_hook))

        # Forward pass (no gradients needed)
        self.eval()
        with torch.no_grad():
            self.forward(data)

        # Remove hooks
        for h in hooks:
            h.remove()

        # Analyse
        report = {}
        for name, act in activations.items():
            # act shape: (batch, neurons)
            # Fraction of batch where each neuron is active (> 0)
            active_frac = (act > 0).float().mean(dim=0)  # shape: (neurons,)
            dead_mask = active_frac <= threshold
            report[name] = {
                "total": act.shape[1],
                "dead": int(dead_mask.sum().item()),
                "dead_fraction": float(dead_mask.float().mean().item()),
                "dead_indices": dead_mask.nonzero(as_tuple=True)[0].tolist(),
            }

        return report

In [19]:
from src.evaluation import load_model, load_data
from config import load_config
from pathlib import Path

exp_dir = Path(r"outputs\dan_750")
eval_data_path = Path(r"C:\Users\cervinka\cervinka\GitHub\MathCAS\datasets\dataset_compressible_flow_5M_test.csv")

config = load_config(exp_dir / "config.yaml")
model = load_model(config, exp_dir)

# Grab at least 10_000 data points across multiple batches
eval_loader = load_data(config, eval_data_path)
batches = []
n_samples = 0
for batch_x, _ in eval_loader:
    batches.append(batch_x)
    n_samples += batch_x.shape[0]
    if n_samples >= 50_000:
        break
data = torch.cat(batches, dim=0)
print(f"Using {data.shape[0]} samples for dying ReLU check")

report = model.check_dying_relus(data)
for layer_name, info in report.items():
    print(f"Layer {layer_name}: {info['dead']}/{info['total']} dead ({info['dead_fraction']:.1%})")

Using 50058 samples for dying ReLU check
Layer 1: 0/750 dead (0.0%)
Layer 3: 26/750 dead (3.5%)
Layer 5: 77/750 dead (10.3%)
Layer 7: 380/750 dead (50.7%)
Layer 9: 425/750 dead (56.7%)
Layer 11: 466/750 dead (62.1%)
Layer 13: 404/750 dead (53.9%)
